# Bài 4.1: Xây dựng và Huấn luyện mô hình Transformer từ đầu (from Scratch)

Notebook này triển khai kiến trúc Transformer gốc theo đúng bài báo "Attention is All You Need" (Vaswani et al., 2017) cho bài toán tóm tắt văn bản tiếng Việt.

In [31]:
# ─── 1. Cài đặt & Import thư viện ──────────────────────────────────────────
import sys
import subprocess

# Tự động cài đặt các thư viện cần thiết nếu chạy trên Kaggle/Colab
pkgs = ["rouge-score", "sacrebleu", "tokenizers", "transformers", "pandas", "pyarrow"]
try:
    import sacrebleu
    import tokenizers
    from rouge_score import rouge_scorer
    print("Các thư viện cần thiết đã có sẵn!")
except ImportError:
    print("⏳ Đang cài đặt các thư viện cần thiết...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
    print("✅ Cài đặt thư viện hoàn tất!")

import os
import math
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị sử dụng: {DEVICE}")


Các thư viện cần thiết đã có sẵn!
Thiết bị sử dụng: cuda


In [32]:
# 2. Đọc dữ liệu & Huấn luyện custom BPE Tokenizer
import os
import pandas as pd
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

# Định nghĩa hàm tự động tìm đường dẫn file dữ liệu trên Kaggle/Colab
def find_parquet(hint="train"):
    """Tìm file parquet theo từ khoá hint trong /kaggle/input hoặc thư mục hiện tại"""
    search_dirs = ["/kaggle/input", ".", "/kaggle/working"]
    for d in search_dirs:
        if not os.path.exists(d):
            continue
        for root, _, files in os.walk(d):
            for f in files:
                if f.endswith(".parquet") and hint.lower() in f.lower():
                    return os.path.join(root, f)
    return None

# Tự động tìm đường dẫn trên Kaggle hoặc local
TRAIN_PATH = find_parquet("train") or "train-00000-of-00001.parquet"
VALID_PATH = find_parquet("valid") or find_parquet("val") or "valid-00000-of-00001.parquet"

print(f"Đường dẫn tìm thấy:")
print(f" - Train: {TRAIN_PATH}")
print(f" - Valid: {VALID_PATH}")

# 2.1 Đọc file parquet dữ liệu
train_df = pd.read_parquet(TRAIN_PATH)
valid_df = pd.read_parquet(VALID_PATH)

# Detect tên cột dữ liệu
src_col = "article" if "article" in train_df.columns else "document"
tgt_col = "summary"
print(f"Cột dữ liệu: nguồn='{src_col}', tóm tắt='{tgt_col}'")

# 2.2 Huấn luyện Tokenizer
all_texts = train_df[src_col].tolist() + train_df[tgt_col].tolist()
print("⏳ Đang huấn luyện BPE Tokenizer từ dữ liệu...")
raw_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
raw_tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"],
    vocab_size=16000
)
raw_tokenizer.train_from_iterator(all_texts, trainer)

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=raw_tokenizer,
    bos_token="[BOS]",
    eos_token="[EOS]",
    unk_token="[UNK]",
    pad_token="[PAD]",
)
print(f"✅ Kích thước từ vựng sau huấn luyện: {tokenizer.vocab_size}")


Đường dẫn tìm thấy:
 - Train: /kaggle/input/datasets/drrins/btl-xlnntn1/train-00000-of-00001.parquet
 - Valid: /kaggle/input/datasets/drrins/btl-xlnntn1/valid-00000-of-00001.parquet
Cột dữ liệu: nguồn='article', tóm tắt='summary'
⏳ Đang huấn luyện BPE Tokenizer từ dữ liệu...



✅ Kích thước từ vựng sau huấn luyện: 16000


In [33]:
# 3. Lập trình kiến trúc Transformer từ đầu (Scratch) 
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, nhead: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % nhead == 0, "d_model must be divisible by nhead"
        self.d_model = d_model
        self.nhead = nhead
        self.d_k = d_model // nhead

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out_linear = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        batch_size = q.size(0)

        q = self.q_linear(q).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)
        k = self.k_linear(k).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)
        v = self.v_linear(v).view(batch_size, -1, self.nhead, self.d_k).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        return self.out_linear(context)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_2(self.dropout(self.activation(self.w_1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, nhead: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        attn_out = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_out))
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, nhead: int, d_ff: int, num_layers: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([EncoderLayer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        x = self.tok_emb(src)
        x = self.pos_emb(x)
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, nhead: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.cross_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor, self_mask: torch.Tensor = None, cross_mask: torch.Tensor = None) -> torch.Tensor:
        attn_out1 = self.self_attn(x, x, x, self_mask)
        x = self.norm1(x + self.dropout1(attn_out1))
        attn_out2 = self.cross_attn(x, enc_out, enc_out, cross_mask)
        x = self.norm2(x + self.dropout2(attn_out2))
        ff_out = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_out))
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, nhead: int, d_ff: int, num_layers: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([DecoderLayer(d_model, nhead, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, trg: torch.Tensor, enc_out: torch.Tensor, self_mask: torch.Tensor = None, cross_mask: torch.Tensor = None) -> torch.Tensor:
        x = self.tok_emb(trg)
        x = self.pos_emb(x)
        for layer in self.layers:
            x = layer(x, enc_out, self_mask, cross_mask)
        return self.norm(x)

class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size: int, trg_vocab_size: int, src_pad_idx: int, trg_pad_idx: int,
                 d_model: int = 256, nhead: int = 8, num_encoder_layers: int = 4, num_decoder_layers: int = 4,
                 d_ff: int = 1024, max_len: int = 512, dropout: float = 0.1, device: str = 'cuda'):
        super().__init__()
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

        self.encoder = Encoder(src_vocab_size, d_model, nhead, d_ff, num_encoder_layers, max_len, dropout)
        self.decoder = Decoder(trg_vocab_size, d_model, nhead, d_ff, num_decoder_layers, max_len, dropout)
        self.generator = nn.Linear(d_model, trg_vocab_size)

    def make_src_mask(self, src: torch.Tensor) -> torch.Tensor:
        return (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)

    def make_trg_mask(self, trg: torch.Tensor) -> torch.Tensor:
        pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(2)
        trg_len = trg.size(1)
        causal_mask = torch.tril(torch.ones((trg_len, trg_len), device=self.device)).bool().unsqueeze(0).unsqueeze(1)
        return pad_mask & causal_mask

    def forward(self, src: torch.Tensor, trg: torch.Tensor) -> torch.Tensor:
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(trg, enc_out, trg_mask, src_mask)
        return self.generator(dec_out)

    @torch.no_grad()
    def generate(self, src: torch.Tensor, max_length: int = 80, bos_token_id: int = 2, eos_token_id: int = 3) -> torch.Tensor:
        self.eval()
        batch_size = src.size(0)
        src_mask = self.make_src_mask(src)
        enc_out = self.encoder(src, src_mask)
        
        trg = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=self.device)
        
        for _ in range(max_length - 1):
            trg_mask = self.make_trg_mask(trg)
            dec_out = self.decoder(trg, enc_out, trg_mask, src_mask)
            logits = self.generator(dec_out[:, -1, :])
            next_tokens = torch.argmax(logits, dim=-1, keepdim=True)
            trg = torch.cat([trg, next_tokens], dim=-1)
        return trg

print("✅ Kiến trúc Transformer from Scratch định nghĩa xong!")

✅ Kiến trúc Transformer from Scratch định nghĩa xong!


In [34]:
# 4. Dataset & DataLoader
class TextSummaryDataset(Dataset):
    def __init__(self, df, tokenizer, src_col, tgt_col, max_input_len=256, max_target_len=80):
        self.df = df
        self.tokenizer = tokenizer
        self.src_col = src_col
        self.tgt_col = tgt_col
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        src_text = str(row[self.src_col])
        tgt_text = str(row[self.tgt_col])

        src_ids = self.tokenizer.encode(src_text, max_length=self.max_input_len, truncation=True)
        tgt_ids = [self.tokenizer.bos_token_id] + self.tokenizer.encode(tgt_text, max_length=self.max_target_len - 2, truncation=True) + [self.tokenizer.eos_token_id]

        return {
            "src_ids": src_ids,
            "tgt_ids": tgt_ids,
            "src_text": src_text,
            "tgt_text": tgt_text
        }

def collate_fn(batch):
    pad_idx = tokenizer.pad_token_id
    src_list, tgt_list = [], []
    src_texts, tgt_texts = [], []
    
    for item in batch:
        src_list.append(torch.tensor(item["src_ids"]))
        tgt_list.append(torch.tensor(item["tgt_ids"]))
        src_texts.append(item["src_text"])
        tgt_texts.append(item["tgt_text"])

    src_padded = nn.utils.rnn.pad_sequence(src_list, batch_first=True, padding_value=pad_idx)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_list, batch_first=True, padding_value=pad_idx)

    return {
        "src": src_padded,
        "tgt": tgt_padded,
        "src_texts": src_texts,
        "tgt_texts": tgt_texts
    }

train_ds = TextSummaryDataset(train_df, tokenizer, src_col, tgt_col)
valid_ds = TextSummaryDataset(valid_df, tokenizer, src_col, tgt_col)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)
print("✅ Khởi tạo DataLoaders thành công!")

✅ Khởi tạo DataLoaders thành công!


In [35]:
# 5. Optimizer & Custom Warmup Scheduler 
class ScheduledOptim:
    def __init__(self, optimizer, d_model, n_warmup_steps):
        self._optimizer = optimizer
        self.n_warmup_steps = n_warmup_steps
        self.n_current_steps = 0
        self.init_lr = np.power(d_model, -0.5)

    def step_and_update_lr(self):
        self._update_learning_rate()
        self._optimizer.step()

    def zero_grad(self):
        self._optimizer.zero_grad()

    def _get_lr_scale(self):
        d_model = self.init_lr
        n_curr_step, n_warmup_step = self.n_current_steps, self.n_warmup_steps
        return np.min([
            np.power(max(n_curr_step, 1), -0.5),
            n_curr_step * np.power(n_warmup_step, -1.5)
        ])

    def _update_learning_rate(self):
        self.n_current_steps += 1
        lr = self.init_lr * self._get_lr_scale()

        for param_group in self._optimizer.param_groups:
            param_group['lr'] = lr

In [36]:
# 6. Bộ đánh giá mô hình
from rouge_score import rouge_scorer
import sacrebleu

class Evaluator:
    def __init__(self):
        self.scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

    def evaluate(self, preds: List[str], refs: List[str]) -> Dict:
        scores = {k: [] for k in ['rouge1', 'rouge2', 'rougeL']}
        for p, r in zip(preds, refs):
            s = self.scorer.score(r, p)
            for k in scores:
                scores[k].append(s[k].fmeasure)
        
        bleu = sacrebleu.corpus_bleu(preds, [refs])
        
        return {
            "rouge1": np.mean(scores["rouge1"]) * 100,
            "rouge2": np.mean(scores["rouge2"]) * 100,
            "rougeL": np.mean(scores["rougeL"]) * 100,
            "bleu": bleu.score
        }

evaluator = Evaluator()
print("✅ Evaluator đã sẵn sàng!")

✅ Evaluator đã sẵn sàng!


In [37]:
# 7. Khởi tạo & Huấn luyện mô hình 
model = Seq2SeqTransformer(
    src_vocab_size=tokenizer.vocab_size,
    trg_vocab_size=tokenizer.vocab_size,
    src_pad_idx=tokenizer.pad_token_id,
    trg_pad_idx=tokenizer.pad_token_id,
    d_model=256,
    nhead=8,
    num_encoder_layers=4,
    num_decoder_layers=4,
    d_ff=1024,
    max_len=256,
    dropout=0.1,
    device=str(DEVICE)
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), betas=(0.9, 0.98), eps=1e-9)
scheduled_optim = ScheduledOptim(optimizer, d_model=256, n_warmup_steps=4000)

print(f"Tổng số tham số của mô hình: {sum(p.numel() for p in model.parameters()):,}")

Tổng số tham số của mô hình: 19,677,824


In [38]:
# 8. Vòng lặp huấn luyện chính
num_epochs = 10
best_rouge2 = 0.0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    start_time = time.time()
    
    for step, batch in enumerate(train_loader):
        src = batch["src"].to(DEVICE)
        tgt = batch["tgt"].to(DEVICE)
        
        # Đầu vào của Decoder là tgt[:, :-1]
        dec_input = tgt[:, :-1]
        # Nhãn mục tiêu là tgt[:, 1:]
        labels = tgt[:, 1:]
        
        scheduled_optim.zero_grad()
        
        logits = model(src, dec_input)
        
        # Tính loss
        loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        scheduled_optim.step_and_update_lr()
        
        total_loss += loss.item()
        
        if (step + 1) % 50 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1:02d} | Step {step+1:04d}/{len(train_loader)} | Loss: {loss.item():.4f} | LR: {current_lr:.2e}")
            
    avg_loss = total_loss / len(train_loader)
    duration = time.time() - start_time
    print(f"\n📢 Epoch {epoch+1:02d} hoàn thành! Loss trung bình: {avg_loss:.4f} | Thời gian: {duration:.1f}s")
    
    # Đánh giá mô hình sau mỗi epoch trên TOÀN BỘ tập validation
    model.eval()
    preds, refs = [], []
    print("⏳ Đang đánh giá trên tập validation...")
    with torch.no_grad():
        for batch in valid_loader:
            src_tensor = batch["src"].to(DEVICE)
            gen_ids = model.generate(
                src_tensor,
                max_length=80,
                bos_token_id=tokenizer.bos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            for i in range(gen_ids.size(0)):
                pred_str = tokenizer.decode(gen_ids[i].tolist(), skip_special_tokens=True)
                preds.append(pred_str)
            refs.extend(batch["tgt_texts"])
            
    metrics = evaluator.evaluate(preds, refs)
    print(f"📊 Đánh giá Epoch {epoch+1:02d}:")
    print(f"  - ROUGE-1: {metrics['rouge1']:.2f}% | ROUGE-2: {metrics['rouge2']:.2f}% | ROUGE-L: {metrics['rougeL']:.2f}%")
    print(f"  - BLEU   : {metrics['bleu']:.2f}%")
    
    # Lưu mô hình tốt nhất
    if metrics['rouge2'] > best_rouge2:
        best_rouge2 = metrics['rouge2']
        torch.save(model.state_dict(), "best_scratch_transformer.pt")
        print("💾 Đã lưu checkpoint mới tốt nhất!")


Epoch 01 | Step 0050/337 | Loss: 9.4986 | LR: 1.24e-05
Epoch 01 | Step 0100/337 | Loss: 9.0715 | LR: 2.47e-05
Epoch 01 | Step 0150/337 | Loss: 8.6837 | LR: 3.71e-05
Epoch 01 | Step 0200/337 | Loss: 8.1767 | LR: 4.94e-05
Epoch 01 | Step 0250/337 | Loss: 7.8440 | LR: 6.18e-05
Epoch 01 | Step 0300/337 | Loss: 7.5998 | LR: 7.41e-05

📢 Epoch 01 hoàn thành! Loss trung bình: 8.5566 | Thời gian: 81.6s
⏳ Đang đánh giá trên tập validation...


📊 Đánh giá Epoch 01:
  - ROUGE-1: 0.01% | ROUGE-2: 0.00% | ROUGE-L: 0.01%
  - BLEU   : 0.01%
Epoch 02 | Step 0050/337 | Loss: 7.3459 | LR: 9.56e-05
Epoch 02 | Step 0100/337 | Loss: 7.2901 | LR: 1.08e-04
Epoch 02 | Step 0150/337 | Loss: 7.1263 | LR: 1.20e-04
Epoch 02 | Step 0200/337 | Loss: 6.9920 | LR: 1.33e-04
Epoch 02 | Step 0250/337 | Loss: 6.9543 | LR: 1.45e-04
Epoch 02 | Step 0300/337 | Loss: 6.7580 | LR: 1.57e-04

📢 Epoch 02 hoàn thành! Loss trung bình: 7.0813 | Thời gian: 80.6s
⏳ Đang đánh giá trên tập validation...


📊 Đánh giá Epoch 02:
  - ROUGE-1: 38.99% | ROUGE-2: 7.14% | ROUGE-L: 24.10%
  - BLEU   : 0.23%
💾 Đã lưu checkpoint mới tốt nhất!
Epoch 03 | Step 0050/337 | Loss: 6.6985 | LR: 1.79e-04
Epoch 03 | Step 0100/337 | Loss: 6.3547 | LR: 1.91e-04
Epoch 03 | Step 0150/337 | Loss: 6.3259 | LR: 2.04e-04
Epoch 03 | Step 0200/337 | Loss: 6.2368 | LR: 2.16e-04
Epoch 03 | Step 0250/337 | Loss: 6.2609 | LR: 2.28e-04
Epoch 03 | Step 0300/337 | Loss: 6.0327 | LR: 2.41e-04

📢 Epoch 03 hoàn thành! Loss trung bình: 6.3538 | Thời gian: 81.0s
⏳ Đang đánh giá trên tập validation...
📊 Đánh giá Epoch 03:
  - ROUGE-1: 40.51% | ROUGE-2: 9.47% | ROUGE-L: 25.01%
  - BLEU   : 0.90%
💾 Đã lưu checkpoint mới tốt nhất!
Epoch 04 | Step 0050/337 | Loss: 5.9897 | LR: 2.62e-04
Epoch 04 | Step 0100/337 | Loss: 5.8963 | LR: 2.74e-04
Epoch 04 | Step 0150/337 | Loss: 5.8171 | LR: 2.87e-04
Epoch 04 | Step 0200/337 | Loss: 5.7535 | LR: 2.99e-04
Epoch 04 | Step 0250/337 | Loss: 5.7565 | LR: 3.12e-04
Epoch 04 | Step 0300/337 | Loss

📊 Đánh giá Epoch 05:
  - ROUGE-1: 46.21% | ROUGE-2: 13.44% | ROUGE-L: 27.74%
  - BLEU   : 1.80%
💾 Đã lưu checkpoint mới tốt nhất!
Epoch 06 | Step 0050/337 | Loss: 5.2916 | LR: 4.29e-04
Epoch 06 | Step 0100/337 | Loss: 5.3004 | LR: 4.41e-04
Epoch 06 | Step 0150/337 | Loss: 5.3980 | LR: 4.53e-04
Epoch 06 | Step 0200/337 | Loss: 5.3490 | LR: 4.66e-04
Epoch 06 | Step 0250/337 | Loss: 5.2407 | LR: 4.78e-04
Epoch 06 | Step 0300/337 | Loss: 5.3063 | LR: 4.90e-04

📢 Epoch 06 hoàn thành! Loss trung bình: 5.2615 | Thời gian: 81.5s
⏳ Đang đánh giá trên tập validation...
📊 Đánh giá Epoch 06:
  - ROUGE-1: 44.85% | ROUGE-2: 14.06% | ROUGE-L: 26.87%
  - BLEU   : 2.07%
💾 Đã lưu checkpoint mới tốt nhất!
Epoch 07 | Step 0050/337 | Loss: 4.9980 | LR: 5.12e-04
Epoch 07 | Step 0100/337 | Loss: 4.8304 | LR: 5.24e-04
Epoch 07 | Step 0150/337 | Loss: 5.0034 | LR: 5.37e-04
Epoch 07 | Step 0200/337 | Loss: 4.9126 | LR: 5.49e-04
Epoch 07 | Step 0250/337 | Loss: 5.0510 | LR: 5.61e-04
Epoch 07 | Step 0300/337 | Lo

In [ ]:
# print("⏳ Đang đánh giá mô hình trên tập Test...")
# model.eval()
# test_preds, test_refs = [], []
# with torch.no_grad():
#     for batch in test_loader:
#         src_tensor = batch["src"].to(DEVICE)
#         gen_ids = model.generate(
#             src_tensor,
#             max_length=80,
#             bos_token_id=tokenizer.bos_token_id,
#             eos_token_id=tokenizer.eos_token_id
#         )
#         for i in range(gen_ids.size(0)):
#             pred_str = tokenizer.decode(gen_ids[i].tolist(), skip_special_tokens=True)
#             test_preds.append(pred_str)
#         test_refs.extend(batch["tgt_texts"])

# test_metrics = evaluator.evaluate(test_preds, test_refs)
# print(f"\n📊 Đánh giá trên tập Test:")
# print(f"  - ROUGE-1: {test_metrics['rouge1']:.2f}% | ROUGE-2: {test_metrics['rouge2']:.2f}% | ROUGE-L: {test_metrics['rougeL']:.2f}%")
# print(f"  - BLEU   : {test_metrics['bleu']:.2f}%")

print(" Cell này dùng để đánh giá trên test set.")

